# Normal Reviews: download places for CA, OR, WA, NV (Overture Maps)

Free and open data, no API key. **Click Runtime → Run all.** It takes roughly **15–30 minutes**; the third box is the slow one.

At the end, **two files** download:
- **west_food.tsv.gz**: restaurants, cafes, bars, bakeries (with names)
- **west_other.tsv.gz**: every other place (location and type only)

Attach **both** to the chat with Claude. If your browser asks to allow multiple downloads, click **Allow**. If one doesn't arrive, click the 📁 folder icon on the left and download it from there.
If anything turns red, copy the error text and paste it to Claude.

In [ ]:
!pip -q install --upgrade duckdb
import duckdb, pandas as pd, re, time
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs; SET s3_region='us-west-2';")
paths = con.sql("SELECT file FROM glob('s3://overturemaps-us-west-2/release/*/theme=places/type=place/*.parquet')").df()["file"]
rel = sorted({re.search(r"release/([^/]+)/", p).group(1) for p in paths})[-1]
SRC = f"s3://overturemaps-us-west-2/release/{rel}/theme=places/type=place/*.parquet"
cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{SRC}')").df()["column_name"].tolist()
print("Overture release:", rel)
print("Columns:", cols)

In [ ]:
# Category: a detailed one (e.g. mexican_restaurant) plus a basic one (e.g. restaurant)
if "taxonomy" in cols:     DETAIL = "taxonomy.primary"
elif "categories" in cols: DETAIL = "categories.primary"
else:                      DETAIL = "basic_category"
BASIC = "basic_category" if "basic_category" in cols else DETAIL
CONF  = "confidence" if "confidence" in cols else "NULL"
CITY  = "addresses[1].locality" if "addresses" in cols else "NULL"
STATE = "addresses[1].region" if "addresses" in cols else "NULL"
OPEN  = "operating_status" if "operating_status" in cols else "NULL"
# Box covering California, Oregon, Washington and Nevada
q = f"""SELECT names.primary AS name, {BASIC} AS basic, {DETAIL} AS detail, {CONF} AS conf,
       (bbox.ymin + bbox.ymax) / 2 AS lat, (bbox.xmin + bbox.xmax) / 2 AS lon,
       {CITY} AS city, {STATE} AS state, {OPEN} AS status
FROM read_parquet('{SRC}')
WHERE bbox.xmin > -124.9 AND bbox.xmax < -114.0 AND bbox.ymin > 32.4 AND bbox.ymax < 49.1"""
t = time.time(); print("Downloading places for CA, OR, WA, NV… (15–30 minutes)")
df = con.sql(q).df()
print(len(df), "places in", round((time.time() - t) / 60, 1), "minutes")

In [ ]:
from google.colab import files
FOOD = (r"restaurant|cafe|lounge|tavern|saloon|taproom|nightlife|coffee|bakery|(^|[_ ])bar($|[_ ])|(^|[_ ])pub($|[_ ])|brewery|brewpub|beer_garden|winery|wine_bar|eatery|bistro|diner|(^|[_ ])deli($|[_ ])|"
        r"sandwich|pizza|taco|burger|sushi|ramen|noodle|dumpling|dim_sum|bagel|donut|doughnut|dessert|ice_cream|frozen_yogurt|gelato|"
        r"juice|smoothie|tea_room|tea_house|bubble_tea|boba|food_court|food_stand|food_truck|creperie|patisserie|pastry|chicken_wings|hot_dog|"
        r"poke|bbq|barbecue|steakhouse|gastropub|tapas|buffet|cafeteria|breakfast_and_brunch|brunch")
NOT = (r"public_|bed_and_breakfast|grocery|supermarket|convenience|liquor|butcher|food_bank|delivery|wholesale|catering|equipment|supply|"
       r"distributor|bartender|barber|bar_association|beauty_bar|nail_bar|oxygen_bar|cooking_school|culinary|cafe_supplier|vending")

d = df.copy()
d = d[d["conf"].isna() | (d["conf"] >= 0.4)]
d = d[~d["status"].astype(str).str.contains("closed", case=False, na=False)]
d["basic"] = d["basic"].astype(str); d["detail"] = d["detail"].fillna(d["basic"]).astype(str)
key = d["detail"] + " " + d["basic"]
is_food = key.str.contains(FOOD, regex=True) & ~key.str.contains(NOT, regex=True)

food = d[is_food & d["name"].notna()][["lat","lon","name","basic","detail","city","state"]].copy()
food["lat"] = food["lat"].round(6); food["lon"] = food["lon"].round(6)
food["name"] = food["name"].astype(str).str.replace("\t", " ").str.replace("\n", " ")
food = food.fillna("").drop_duplicates()

other = d[~is_food & d["basic"].ne("None")][["lat","lon","basic"]].copy().round({"lat":5,"lon":5}).drop_duplicates()

print(len(food), "food places. By state:"); print(food["state"].value_counts().head(8))
print(food["detail"].value_counts().head(20))
print(len(other), "other places. Top types:"); print(other["basic"].value_counts().head(12))
food.to_csv("west_food.tsv.gz", sep="\t", index=False, compression="gzip")
other.to_csv("west_other.tsv.gz", sep="\t", index=False, compression="gzip")
!ls -lh west_food.tsv.gz west_other.tsv.gz
files.download("west_food.tsv.gz")
import time as _t; _t.sleep(4)
files.download("west_other.tsv.gz")